In [0]:
# ============================================================
# Project  : ABC E-Commerce
# Layer    : Silver
# Notebook : 06_Silver_Customers
# Author   : Suriya
# Purpose  : Clean and validate customer data
# ============================================================

In [0]:
from pyspark.sql.functions import *

In [0]:
customers_df = spark.table("workspace.default.bronze_customers")

In [0]:
customers_df.show(5)

customers_df.printSchema()

print("Total Records :", customers_df.count())

In [0]:
null_records = customers_df.filter(
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("email").isNull()
)

print("Null Records :", null_records.count())

null_records.show()

In [0]:
valid_df = customers_df.filter(
    col("customer_id").isNotNull() &
    col("customer_name").isNotNull() &
    col("email").isNotNull()
)

In [0]:
duplicate_df = (
    valid_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_df.show()

In [0]:
silver_df = valid_df.dropDuplicates(["customer_id"])

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.silver_customers")

In [0]:
silver_df = (
    silver_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("data_source", lit("customers.csv"))
)

In [0]:
silver_df.show(5)

silver_df.printSchema()

print("Silver Record Count :", silver_df.count())

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.silver_customers")

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_customers
LIMIT 10
""").show()